# Missing Value Analysis

This notebook analyzes the ~300 incomplete vulnerability records and evaluates whether they should be preserved, imputed, or treated through a more explicit missing-value strategy.

Important rule:
- Do not auto-fill CVSS or EPSS with zero.
- Do not delete the incomplete records without justification.

In [ ]:
# Setup cell: Colab-compatible project setup.
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()

if "google.colab" in sys.modules:
    repo_url = os.getenv("GITHUB_REPO_URL")
    if repo_url:
        !git clone {repo_url} /content/cyberguard-ai
        %cd /content/cyberguard-ai
        REPO_ROOT = Path.cwd()

for candidate in [REPO_ROOT, Path("/content/cyberguard-ai"), Path("/workspace"), Path("/content")]:
    if (candidate / "data" / "processed" / "risk_features.csv").exists():
        REPO_ROOT = candidate
        break

os.chdir(REPO_ROOT)
print(f"Project root: {REPO_ROOT}")

!python -m pip install -q pandas numpy matplotlib plotly scikit-learn

import pandas as pd

DATA_PATH = REPO_ROOT / "data" / "processed" / "risk_features.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape}")

In [ ]:
# Missingness overview.
missing = df.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_percent[missing_percent > 0])

In [ ]:
# Check correlation of missingness patterns.
missing_flag = df.isna().astype(int)
print(missing_flag.head())
print('\nMissingness co-occurrence:\n')
print(missing_flag.corr().round(3).to_string())

In [ ]:
# Investigate the incomplete records.
incomplete = df[df[["CVSS_Score", "EPSS_Score", "Severity", "CVSS_Version", "EPSS_Percentile", "Severity_Encoded"]].isna().any(axis=1)].copy()
print(f"Incomplete rows: {len(incomplete)}")
print(incomplete.head().to_string())

for col in ["CVE_ID", "Description", "Published_Date", "KEV", "Severity", "CVSS_Score", "EPSS_Score"]:
    if col in incomplete.columns:
        print(f"\nColumn: {col}")
        print(incomplete[col].head())

In [ ]:
# Assess whether CVSS and EPSS are missing together.
cvss_missing = df["CVSS_Score"].isna()
epss_missing = df["EPSS_Score"].isna()
print('CVSS missing count:', cvss_missing.sum())
print('EPSS missing count:', epss_missing.sum())
print('CVSS and EPSS both missing:', (cvss_missing & epss_missing).sum())
print('CVSS missing but EPSS present:', (cvss_missing & ~epss_missing).sum())
print('EPSS missing but CVSS present:', (epss_missing & ~cvss_missing).sum())

In [ ]:
# Check whether KEV and descriptions remain available in incomplete rows.
print('KEV available in incompletes:', incomplete['KEV'].notna().sum())
print('Description available in incompletes:', incomplete['Description'].notna().sum())
print('Published date available in incompletes:', incomplete['Published_Date'].notna().sum())
print('KEV values in incompletes:', incomplete['KEV'].value_counts(dropna=False).to_dict())

In [ ]:
# Compare the incomplete rows to the complete rows on key attributes.
complete = df[df["CVSS_Score"].notna() & df["EPSS_Score"].notna()].copy()
for col in ["KEV", "Vulnerability_Age_Days", "Severity_Encoded"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print('Incomplete mean:', incomplete[col].mean(skipna=True) if pd.api.types.is_numeric_dtype(incomplete[col]) else 'n/a')
        print('Complete mean:', complete[col].mean(skipna=True) if pd.api.types.is_numeric_dtype(complete[col]) else 'n/a')

## Missingness strategies

Compare three options:

A. Preserve missing values and allow LightGBM to handle them.
B. Impute with explicit missing indicators.
C. Use a statistically justified method such as median imputation for numeric features and a separate missing-category indicator for categorical fields.

The recommended strategy is not to set CVSS or EPSS to zero.

In [ ]:
# Save a summary file for documentation.
summary_text = """
Missing Value Analysis Summary
=============================

Total rows: {rows}
Incomplete rows: {incomplete_rows}

Missing columns and counts:
{missing_counts}

CVSS and EPSS pattern:
- CVSS missing: {cvss_missing}
- EPSS missing: {epss_missing}
- Both missing: {both_missing}
- CVSS only missing: {cvss_only_missing}
- EPSS only missing: {epss_only_missing}

KEV and descriptions in incomplete rows:
- KEV available: {kev_available}
- Description available: {desc_available}
- Published date available: {pub_available}

Recommendation:
- Preserve the missingness pattern rather than zero-filling CVSS/EPSS.
- For ML training, use a preprocessing strategy that supports missing values explicitly.
- Keep the missing records unless a separate, documented temporal exclusion criterion is justified.
""".format(
    rows=len(df),
    incomplete_rows=len(incomplete),
    missing_counts=df.isna().sum().sort_values(ascending=False).to_string(),
    cvss_missing=int(df['CVSS_Score'].isna().sum()),
    epss_missing=int(df['EPSS_Score'].isna().sum()),
    both_missing=int((df['CVSS_Score'].isna() & df['EPSS_Score'].isna()).sum()),
    cvss_only_missing=int((df['CVSS_Score'].isna() & df['EPSS_Score'].notna()).sum()),
    epss_only_missing=int((df['EPSS_Score'].isna() & df['CVSS_Score'].notna()).sum()),
    kev_available=int(incomplete['KEV'].notna().sum()),
    desc_available=int(incomplete['Description'].notna().sum()),
    pub_available=int(incomplete['Published_Date'].notna().sum()),
)

report_path = REPO_ROOT / 'reports' / 'missing_value_analysis.md'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(summary_text, encoding='utf-8')
print(f'Saved: {report_path}')